# Triplet Network Evaluation

## Import Libraries

In [ ]:
!pip install datasets --no-warn-conflicts --quiet

import os
import datasets
import numpy as np
import random
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
import torch
from PIL import Image
import torchvision
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 20.8 MB/s eta 0:00:00


## Configuration

In [ ]:
import os
from pathlib import Path
import torch


class Config:
    """
    Configuration class for the project.
    """
    # ========== Base Paths ==========
    project_name = "logo_recognition_similarity_search_project"
    base_dir = Path("/content/drive/MyDrive")/project_name
    output_dir = base_dir / "output"
    cache_dir = base_dir / "cache"

    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(cache_dir, exist_ok=True)

    # ========== Dataset ==========
    dataset_path = "mlproject5606/Logo-Recognition-Triplet-Indices-Train-Dataset"
    dataset_cache_dir = cache_dir / dataset_path.split("/")[-1]
    category_indices_cache_dir = cache_dir / "Category-Indices"
    train_category_indices_path = category_indices_cache_dir / "category_train_indices.json"
    test_category_indices_path =  category_indices_cache_dir / "category_test_indices.json"


    # ========== Runtime ==========
    device = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers = 4 if os.cpu_count()>=4 else 0
    seed = 42

    # ========== Model ==========
    model_path = 'mlproject5606/Logo-Recognition-ResNet50-Triplet-Retrieval'
    output_model_dir = output_dir / model_path.split("/")[-1]

    # ========== Training ==========
    input_dim = 2048
    embed_dim = 256
    batch_size = 1024  #512,2048,1024
    margin = 1.0
    p = 2
    reduction='mean' #  'none' | 'mean' | 'sum'
    lr = 0.001
    scheduler_step_size = 2
    scheduler_gamma = 0.1
    num_epochs = 10
    save_every = 1



# Create Configuration Object
cfg = Config()

## Load Dataset

In [ ]:
from datasets import load_dataset

ds_train = load_dataset(path = cfg.dataset_path,
                       split = "train",
                        )

ds_train

In [ ]:
# from datasets import load_from_disk

# ds_train = load_from_disk(cfg.dataset_cache_dir)
# ds_train

In [ ]:
# Inspect the structure of the dataset
ds_train.features

{'category': Value(dtype='int32', id=None),
 'resnet50_embedding': Sequence(feature=Value(dtype='float64', id=None), length=-1, id=None),
 'resnet50_class': [{'label': Value(dtype='string', id=None),
   'score': Value(dtype='float64', id=None)}],
 'index': Value(dtype='int64', id=None),
 'positive_idx': Value(dtype='int64', id=None),
 'negative_idx': Value(dtype='int64', id=None),
 'negative_category': Value(dtype='int64', id=None)}

## Triplet Dataset & DataLoader

In [ ]:
import os
import json
import torch
from torch.utils.data import Dataset
from collections import defaultdict
from tqdm import tqdm
import random

class TripletLogoDataset(Dataset):
    def __init__(self,ds: datasets.Dataset, category_indices_path:str,seed:int=None):
        """
        Initialize the TripletLogoDataset.
        """
        self.ds = ds
        self.rng = random.Random() if seed is None else random.Random(seed)
        self.category_indices_path = category_indices_path
        self.category_to_indices = self.load_or_create_category_indices(ds, category_indices_path)
        self.categories = list(self.category_to_indices.keys())



    def __len__(self):
        """
        Return the number of samples in the dataset.
        """
        return len(self.ds)

    def _get_positive_index(self, anchor_idx, category):
        """
        Get a positive index for the given anchor index and category.
        """
        positive_indices = self.category_to_indices[category].copy()
        positive_indices.remove(anchor_idx)
        if not positive_indices:
            print(f"No positive indices found for category {category}")
            return anchor_idx
        positive_idx = self.rng.choice(positive_indices)
        return positive_idx

    def _get_negative_index(self, anchor_idx, category):
        """
        Get a negative index for the given anchor index and category.
        """
        negative_categories = self.categories.copy()
        negative_categories.remove(category)
        negative_category = self.rng.choice(negative_categories)
        negative_indices = self.category_to_indices[negative_category]
        negative_idx = self.rng.choice(negative_indices)
        return negative_idx


    def config_triplet_dataset(self):
        """
        Configures the dataset for triplet training.
        """
        self.ds = self.ds.map(process_example,
                              with_indices=True,
                              desc="Processing examples for triplet training",
                              )

        def process_example(example,idx):
            example['index'] = idx
            example['resnet50_embedding'] = np.array(example['resnet50_embedding']).reshape(-1)
            category = example["category"]
            category = category.item() if isinstance(category, torch.Tensor) else category
            example['positive_idx'] = self._get_positive_index(idx, category)
            example['negative_idx'] = self._get_negative_index(idx, category)
            return example


    def __getitem__(self, idx):
        """
        Get a triplet sample from the dataset.
        """
        anchor_idx = idx
        anchor = self.ds[anchor_idx]
        postive_idx = anchor['positive_idx']
        negative_idx = anchor['negative_idx']
        positive = self.ds[postive_idx]
        negative = self.ds[negative_idx]

        anchor = torch.tensor(anchor['resnet50_embedding'])
        positive = torch.tensor(positive['resnet50_embedding'])
        negative = torch.tensor(negative['resnet50_embedding'])

        return anchor, positive, negative


    def collate_fn(self,batch):
        """
        Collate function for the DataLoader.
        """
        idxs = list(map(lambda x: (x['positive_idx'],x['negative_idx']),batch))
        positive_indices, negatives_indices = zip(*idxs)

        anchors = torch.stack([torch.tensor(x["resnet50_embedding"]) for x in batch])
        positives = torch.stack([torch.tensor(x) for x in self.ds.select(positive_indices)['resnet50_embedding']])
        negatives = torch.stack([torch.tensor(x) for x in self.ds.select(negatives_indices)['resnet50_embedding']])

        return anchors, positives, negatives



    @staticmethod
    def load_or_create_category_indices(ds, file_path):
        """
        Load category_to_indices mapping from file if exists,
        otherwise create it from the dataset and save as JSON.
        """
        category_indices_cache_dir = os.path.dirname(file_path)
        os.makedirs(category_indices_cache_dir, exist_ok=True)

        if os.path.exists(file_path):
            print(f"[INFO] Loading category_to_indices from: {file_path}")
            with open(file_path, 'r') as f:
                category_to_indices = json.load(f)
                category_to_indices = {int(k): v for k, v in category_to_indices.items()}
                return category_to_indices
        else:
            print(f"[INFO] Creating category_to_indices from dataset...")
            category_to_indices = defaultdict(list)
            for idx, example in enumerate(tqdm(ds, desc="Building category to indices")):
                category_to_indices[example['category']].append(idx)

            with open(file_path, 'w') as f:
                json.dump(category_to_indices, f)
            print(f"[INFO] Saved category_to_indices to: {file_path}")
            return category_to_indices


    @staticmethod
    def split_dataset_by_category_indices(ds, category_to_indices, test_ratio=0.1, seed=42):
        """
        Split dataset into train/test indices by category.
        """
        random.seed(seed)

        train_indices = []
        test_indices = []

        for cat, indices in tqdm(category_to_indices.items(), desc="Splitting train/test"):
            if len(indices) <2:
                continue

            indices_copy = indices.copy()
            random.shuffle(indices_copy)

            test_count = max(1, int(len(indices_copy) * test_ratio))

            if len(indices_copy)-test_count < 2:
                train = indices_copy[:]
                test = []
                print(f"Skipping category {cat} due to insufficient images in test set")
            else:
                test = indices_copy[:test_count]
                train = indices_copy[test_count:]

            train_indices.extend(train)
            test_indices.extend(test)

        ds_train = ds.select(train_indices)
        ds_test = ds.select(test_indices)

        return ds_train, ds_test

In [ ]:
train_dataset = TripletLogoDataset(ds = ds_train,
                                   category_indices_path=cfg.train_category_indices_path,
                                   seed=None
                                  )

print(f"[INFO] Number of samples in train_dataset: {len(train_dataset)}")

[INFO] Loading category_to_indices from: /content/drive/MyDrive/logo_recognition_similarity_search_project/cache/Category-Indices/category_train_indices.json
[INFO] Number of samples in train_dataset: 1458113


In [ ]:
# Configures the dataset for triplet training.
# train_dataset.config_triplet_dataset()

In [ ]:
train_loader  = torch.utils.data.DataLoader(train_dataset.ds,
                                            batch_size=cfg.batch_size,
                                            num_workers=cfg.num_workers,
                                            pin_memory=True,
                                            collate_fn = train_dataset.collate_fn
                                            )
print(f"[INFO] Number of batches in train_loader: {len(train_loader)}")

[INFO] Number of batches in train_loader: 1424


In [ ]:
# %%time
# batch = next(iter(train_loader))

In [ ]:
# %%time
# anchors, positives, negatives = batch
# print(f"Shape of anchors: {anchors.shape}")
# print(f"Shape of positives: {positives.shape}")
# print(f"Shape of negatives: {negatives.shape}")

## Evaluation